# Institution Resolver v3 — Qwen3-4B Hakem (Judge) Colab Notebook

Bu notebook, `data/jobs/inventory_sonuc.csv`'deki `needs_review=1` satirlarini (gate'in karar veremedigi, LLM hakeme dusen sorgular) `qwen3:4b-instruct-2507-q4_K_M` modeliyle Colab GPU uzerinde islemek icin hazirlanmistir.

**`notebooks/colab_run_280k.ipynb`'a (Gemma 4 E4B, gate-only 280k) dokunulmadi — bu tamamen ayri, bagimsiz bir notebook.**

### Onemli fark (neden ayri notebook):
- Model: `gemma4:e4b` (~8B, ozel Ollama import gerektirir) yerine `qwen3:4b-instruct-2507-q4_K_M` (~4B, **herkese acik Ollama registry'sinde** — `ollama pull` ile dogrudan iner, Drive'a onceden yukleme gerekmez).
- Kapsam: 280k'nin tamami degil, sadece `needs_review=1` alt kumesi (~149k, tam sayi yerel koşu bitince netlesir) — cunku LLM hakem SADECE gate'in karar veremedigi satirlara gidiyor.
- Amac: Gate degil, **hakem katmani** hizi + karar kalitesi.

### On kosul (yerelde, bu notebook'tan ONCE yapilmali):
`data/jobs/inventory_sonuc.csv` (yerel gate-only kosunun ciktisi) tamamlandiktan/yeterince ilerledikten sonra, `needs_review=1` satirlarini kucuk bir CSV'ye ayiklayip Drive'a yuklemeniz gerekiyor:

```bash
python3 - <<'EOF'
import csv
with open('data/jobs/inventory_sonuc.csv', newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r.get('needs_review') == '1']
with open('data/jobs/needs_review_subset.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['query', 'normalized_name', 'rows'])
    w.writeheader()
    for r in rows:
        w.writerow({'query': r['query'], 'normalized_name': r.get('normalized_name', ''), 'rows': r.get('rows', '')})
print(len(rows), 'satir yazildi -> data/jobs/needs_review_subset.csv')
EOF
```

Sonra `needs_review_subset.csv`'yi Google Drive'daki `institution_resolver_v3/jobs/` klasorune yukleyin (bu notebook'un 2. hucresi ayni Drive kok dizinini kullaniyor, `colab_run_280k.ipynb` ile paylasilan `DRIVE_JOBS` klasoru — sadece dosya adi farkli, cakisma yok).

## 1) Donanim & GPU Kontrolu

In [ ]:
!nvidia-smi

## 2) Google Drive Baglama + Dizin Yollari

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/institution_resolver_v3"
DRIVE_RAW = f"{DRIVE_ROOT}/data_raw"
DRIVE_PROCESSED = f"{DRIVE_ROOT}/data_processed"
DRIVE_JOBS = f"{DRIVE_ROOT}/jobs"
DRIVE_EVAL = f"{DRIVE_ROOT}/data_eval"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"

for p in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_JOBS, DRIVE_EVAL, DRIVE_OUTPUT):
    os.makedirs(p, exist_ok=True)

print("Drive klasorleri hazir:", DRIVE_ROOT)
print("Beklenen girdi dosyasi:", f"{DRIVE_JOBS}/needs_review_subset.csv", "- yuklendi mi kontrol edin.")

## 3) Koda Erisim (Git Clone / Pull)

In [ ]:
REPO_DIR = "/content/institution_resolver_v3"
BRANCH = "feat/gate-asama1"

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}

## 4) Drive Sembolik Linkleri

In [ ]:
import os, shutil

def _link(name, target):
    os.makedirs(target, exist_ok=True)
    link = f"data/{name}"
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        for f in os.listdir(link):
            src, dst = f"{link}/{f}", f"{target}/{f}"
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(link)
    os.symlink(target, link)

os.makedirs("data", exist_ok=True)
_link("raw", DRIVE_RAW)
_link("processed", DRIVE_PROCESSED)
_link("jobs", DRIVE_JOBS)

!ls -la data
!ls -la data/jobs/needs_review_subset.csv

## 5) Elasticsearch Kurulumu ve Baslatma

Hakem, gate'in urettigi aday havuzuna (candidate list) ihtiyac duyuyor - o yuzden judge-only bir kosuda bile ES + indeks sarttir (resolve() her sorguda yeniden calisir).

In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es
  tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi

grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF

cat > /content/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/content/es/config/elasticsearch.policy" \
  setsid /content/es/bin/elasticsearch < /dev/null > /content/es/es.log 2>&1 &
disown
sleep 3

In [ ]:
import time, requests

for _ in range(60):
    try:
        r = requests.get("http://localhost:9200/_cluster/health", timeout=2)
        if r.status_code == 200:
            print("Elasticsearch Saglikli:", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ES baslatilamadi, /content/es/es.log kontrol edin")

## 6) Ollama Kurulumu + Qwen3-4B (dogrudan `ollama pull`, Drive'a onceden yukleme gerekmez)

`qwen3:4b-instruct-2507-q4_K_M`, `gemma4:e4b`'nin aksine herkese acik Ollama registry'sinde - Colab'in hizli baglantisiyla (~2,5 GB) dogrudan iniyor. `OLLAMA_KEEP_ALIVE=-1` ile model GPU'dan hic dusurulmuyor.

In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd

if ! command -v ollama &> /dev/null; then
  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
fi

ollama --version

In [ ]:
import os, subprocess, time, requests

MODEL_TAG = "qwen3:4b-instruct-2507-q4_K_M"

# NUM_PARALLEL=8 (orijinal 280k/gemma notebook'undan miras) + num_ctx=8192
# GPU VRAM'inin neredeyse tamamini KV-cache'e ayiriyordu (canli olcum, 2026-08-12:
# 40 GB'lik A100'de 38+ GB Ollama'ya gitti), embedding modeline (sentence-
# transformers, ayni GPU'yu paylasiyor) yer kalmiyor, CUDA OOM cikiyordu.
# 2'ye dusurmek KV-cache payini ~4x kucultup ikisine de yer birakiyor.
os.environ["OLLAMA_NUM_PARALLEL"] = "2"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"  # ASLA VRAM'dan dusurme

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
    env=os.environ,
)

for _ in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            print("Ollama servisi baslatildi (KEEP_ALIVE=-1, NUM_PARALLEL=2)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama baslatilamadi")

!ollama pull {MODEL_TAG}

print("Model VRAM'a on-yukleniyor (warm-up)...")
r = requests.post("http://localhost:11434/api/generate", json={
    "model": MODEL_TAG,
    "prompt": "Merhaba",
    "stream": False,
})
if r.status_code == 200:
    print("Model GPU belleginde sicak ve hazir!")
else:
    print("UYARI: Warm-up basarisiz, log kontrol edin:", r.text[:200])

## 7) Python Paket Kurulumu

In [ ]:
!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e ".[dev,embed,llm,api]"

## 8) Elasticsearch Sema Sifirlama ve Indeksleme

In [ ]:
!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings

## 9) Tekli Sorgu Testi (Her Sey Calisiyor mu?)

Yerelde bes bilinen sorguda (Milli Egitim Bakanligi, Diyanet Isleri Baskanligi, MEB, ...) Qwen'in Gemma ile ayni kararlari verdigini zaten gorduk (5/5) - burada tek sorguyla pipeline'in Colab'de de dogru calistigini dogruluyoruz.

In [ ]:
!python3 -m institution_resolver_v3.cli.main judge "Milli Egitim Bakanligi" --model "qwen3:4b-instruct-2507-q4_K_M"

## 10) Kucuk Olcekte Hiz + Kalite Testi (tam kosuya gecmeden once)

`needs_review_subset.csv`'den ilk 100 satiri, farkli worker sayilariyla, GERCEK hakem cagrisiyla (judge dahil) test eder. `colab_run_280k.ipynb`'daki benchmark sadece gate'i test ediyordu - burada asil ilgilendigimiz hakem hizi oldugu icin `inventory-batch --judge` ile gercek uctan uca zamanlama aliniyor.

In [ ]:
INPUT_CSV = f"{DRIVE_JOBS}/needs_review_subset.csv"
TEST_OUTPUT = "/content/qwen_judge_test100.csv"

import time
t0 = time.time()
!python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --model "qwen3:4b-instruct-2507-q4_K_M" --out "{TEST_OUTPUT}" --limit 100 --workers 2
dt = time.time() - t0
print(f"\n100 satir (hakem dahil) toplam sure: {dt:.1f} sn -> {dt/100:.2f} sn/sorgu")
print("149.000 satir icin kaba tahmin:", f"{149000 * dt/100 / 3600:.1f} saat")

## 10.5) İşçi (`--workers`) Sayısı Testi

Aynı 30 `needs_review` satırını farklı `--workers` değerleriyle, GERÇEK hakem çağrısıyla, bağımsız kosular halinde çalıştırıp karşılaştırır. `OLLAMA_NUM_PARALLEL` sabit (2) kalıyor — VRAM tüketimini asıl o belirliyor, istemci tarafındaki `--workers` sayısı GPU belleğini etkilemiyor (sadece Ollama'nın 2 slotuna kaç istemci kuyruklanacağını), o yüzden bu testte OOM riski yok. Amaç: kaç işçiden sonra kazanç düzleşiyor/geriliyor, onu bulmak.

In [ ]:
import os, csv, time

INPUT_CSV = f"{DRIVE_JOBS}/needs_review_subset.csv"
WORKER_COUNTS = [1, 2, 4, 8, 16]
TEST_LIMIT = 30  # kucuk tut, her worker sayisi ayri kosu oldugu icin toplam sure carpanli artiyor

print(f"ISCI HIZ TESTI (hakem dahil, ilk {TEST_LIMIT} needs_review satiri, her worker sayisi icin BAGIMSIZ kosu)...\n")

results = []
for workers in WORKER_COUNTS:
    out = f"/content/worker_test_w{workers}.csv"
    if os.path.exists(out):
        os.remove(out)

    t0 = time.time()
    !python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --model "qwen3:4b-instruct-2507-q4_K_M" --out "{out}" --limit {TEST_LIMIT} --workers {workers} > /content/worker_test_w{workers}.log 2>&1
    dt = time.time() - t0

    with open(out, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    ok = sum(1 for r in rows if r.get("status") == "ok")
    qps = TEST_LIMIT / dt if dt > 0 else 0
    results.append((workers, dt, dt / TEST_LIMIT, qps, ok))
    print(f"workers={workers:>2d} -> toplam={dt:6.1f}s | {dt/TEST_LIMIT:5.2f} sn/sorgu | {qps:5.2f} sorgu/sn | basarili={ok}/{TEST_LIMIT}")

print("\n--- OZET ---")
best = min(results, key=lambda r: r[2])
print(f"En hizli: workers={best[0]} -> {best[2]:.2f} sn/sorgu")
print(f"149.000 satir icin bu workers ile tahmini sure: {149000 * best[2] / 3600:.1f} saat")
print("\nNot: sn/sorgu duzlesmeye/kotulesmeye basladigi noktadan sonraki degerler")
print("Ollama'nin NUM_PARALLEL=2 slotunda kuyruklanmadan geliyor - kazanc yoksa artirmayin.")

## 11) TAM KOSU: needs_review Alt Kumesi + Qwen3-4B Hakem

Yukaridaki test sonucunu (hiz + `TEST_OUTPUT`'taki kararlarin kalitesini) gozden gecirdikten sonra calistirin. `--resume` ile kesinti/yeniden baslatmada kaldigi yerden devam eder; Colab oturumu koparsa, bu hucreyi tekrar calistirmak yeterli (Drive'daki `LOCAL_OUTPUT` kopyasi resume icin kullanilir).

In [ ]:
import os, shutil

INPUT_CSV = f"{DRIVE_JOBS}/needs_review_subset.csv"
LOCAL_OUTPUT = "/content/qwen_judge_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/qwen_judge_sonuc.csv"
MODEL_TAG = "qwen3:4b-instruct-2507-q4_K_M"

# Varsa onceden islenen kismi yerel diske kopyala (resume icin):
if os.path.exists(DRIVE_FINAL_OUTPUT) and not os.path.exists(LOCAL_OUTPUT):
    shutil.copy(DRIVE_FINAL_OUTPUT, LOCAL_OUTPUT)

print("needs_review + Qwen3-4B hakem kosusu basliyor...")

!python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --model "{MODEL_TAG}" --out "{LOCAL_OUTPUT}" --workers 2 --resume

shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
print("ISLEM BITTI! Sonuc Google Drive'a kopyalandi:", DRIVE_FINAL_OUTPUT)

## 12) (Opsiyonel) Uzun kosu sirasinda ara yedekleme

Colab oturumlari saatler surdugunde araya bir hucre daha eklemek isterseniz, bu hucreyi ayri calistirarak `LOCAL_OUTPUT`'u periyodik olarak Drive'a yedekleyebilirsiniz (11. hucre bitene kadar beklemeden, baska bir sekmede).

In [ ]:
import shutil, time

LOCAL_OUTPUT = "/content/qwen_judge_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/qwen_judge_sonuc.csv"

shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
print("Ara yedek alindi:", time.strftime("%H:%M:%S"))